# Chapter 4 — Quantum Error Correction

**Learning objectives**
- Understand why classical error correction ideas fail for quantum states
- Implement the three-qubit bit-flip code
- Implement the three-qubit phase-flip code
- Combine them into the nine-qubit Shor code
- Understand stabilizer formalism and syndrome measurement
- Connect code distance to physical qubit overhead via the resource estimator

## Setup

In [ ]:
import qsharp
import numpy as np
import matplotlib.pyplot as plt
print(f"qsharp {__import__("importlib.metadata", fromlist=["version"]).version("qsharp")}")

## 4.1 Classical vs Quantum Errors

Classical bits can be protected by redundancy: send `0` as `000` and majority-vote on readout. Quantum states resist direct copying (no-cloning theorem) and direct measurement collapses the superposition. QEC works differently:

1. **Encode** one logical qubit into many physical qubits
2. **Detect** errors by measuring *stabilizers* (parity checks that commute with the logical state)
3. **Correct** by applying a recovery operation based on the syndrome (error signature)

Only X (bit-flip), Z (phase-flip), and Y=XZ errors need to be corrected — an arbitrary single-qubit error can be decomposed in the Pauli basis.

## 4.2 Three-Qubit Bit-Flip Code

Encodes α|0⟩ + β|1⟩ as α|000⟩ + β|111⟩.  
Syndrome: measure parity of (q0,q1) and (q1,q2). Odd parity → error on that qubit.

Reference: `samples/algorithms/BitFlipCode.qs`

In [ ]:
%%qsharp

import Std.Math.*;
import Std.Diagnostics.*;

/// Encode a single qubit into a 3-qubit bit-flip code
operation EncodeBitFlip(physQ : Qubit, aux : Qubit[]) : Unit is Adj {
    // α|0⟩ + β|1⟩  →  α|000⟩ + β|111⟩
    CNOT(physQ, aux[0]);
    CNOT(physQ, aux[1]);
}

/// Measure syndrome and return (parity01, parity12)
operation MeasureBitFlipSyndrome(qs : Qubit[]) : (Result, Result) {
    use ancilla = Qubit[2];
    CNOT(qs[0], ancilla[0]);
    CNOT(qs[1], ancilla[0]);
    CNOT(qs[1], ancilla[1]);
    CNOT(qs[2], ancilla[1]);
    let s = (M(ancilla[0]), M(ancilla[1]));
    ResetAll(ancilla);
    s
}

/// Correct a single bit-flip error given syndrome
operation CorrectBitFlip(qs : Qubit[], syndrome : (Result, Result)) : Unit {
    let (s01, s12) = syndrome;
    let errorIdx =
        if (s01, s12) == (One, Zero) { 0 }
        elif (s01, s12) == (One, One) { 1 }
        elif (s01, s12) == (Zero, One) { 2 }
        else { -1 };
    if errorIdx >= 0 { X(qs[errorIdx]); }
}

/// End-to-end bit-flip code demo with injected error
operation BitFlipCodeDemo(errorQubit : Int) : Result {
    use qs = Qubit[3];
    // Prepare a non-trivial state (20/80 superposition)
    let alpha = 0.20;
    Ry(2.0 * ArcCos(Sqrt(alpha)), qs[0]);

    EncodeBitFlip(qs[0], qs[1..2]);

    // Inject a bit-flip error
    if errorQubit >= 0 { X(qs[errorQubit]); }

    Message($"State after error on qubit {errorQubit}:");
    DumpMachine();

    let syndrome = MeasureBitFlipSyndrome(qs);
    Message($"Syndrome: {syndrome}");
    CorrectBitFlip(qs, syndrome);

    Message("State after correction:");
    DumpMachine();

    // Decode and measure
    Adjoint EncodeBitFlip(qs[0], qs[1..2]);
    let r = M(qs[0]);
    ResetAll(qs);
    r
}

In [ ]:
# Run the demo with each possible error location
for err in [-1, 0, 1, 2]:
    label = "no error" if err < 0 else f"error on qubit {err}"
    results = qsharp.run(f"BitFlipCodeDemo({err})", 20)
    from collections import Counter
    counts = Counter(str(r) for r in results)
    print(f"  {label:20s}: {dict(counts)}")

## 4.3 Three-Qubit Phase-Flip Code

Phase-flip (Z) errors are invisible in the Z basis but appear as X errors in the Hadamard basis. The phase-flip code encodes α|+⟩ + β|−⟩ = α|+++⟩ + β|−−−⟩ (logically).

Reference: `samples/algorithms/PhaseFlipCode.qs`

In [ ]:
%%qsharp

import Std.Math.*;
import Std.Diagnostics.*;

/// Encode into 3-qubit phase-flip code (H-basis bit-flip code)
operation EncodePhaseFlip(physQ : Qubit, aux : Qubit[]) : Unit is Adj {
    // Bit-flip encoding, then change basis
    CNOT(physQ, aux[0]);
    CNOT(physQ, aux[1]);
    // Change from Z to X basis: |0⟩ → |+⟩, |1⟩ → |−⟩
    H(physQ); H(aux[0]); H(aux[1]);
}

operation MeasurePhaseFlipSyndrome(qs : Qubit[]) : (Result, Result) {
    use ancilla = Qubit[2];
    // Measure Z-basis parity in X basis = X-basis parity
    // Change to Z basis, check, change back
    ApplyToEach(H, qs);
    CNOT(qs[0], ancilla[0]);
    CNOT(qs[1], ancilla[0]);
    CNOT(qs[1], ancilla[1]);
    CNOT(qs[2], ancilla[1]);
    ApplyToEach(H, qs);
    let s = (M(ancilla[0]), M(ancilla[1]));
    ResetAll(ancilla);
    s
}

operation CorrectPhaseFlip(qs : Qubit[], syndrome : (Result, Result)) : Unit {
    let (s01, s12) = syndrome;
    let errorIdx =
        if (s01, s12) == (One, Zero) { 0 }
        elif (s01, s12) == (One, One) { 1 }
        elif (s01, s12) == (Zero, One) { 2 }
        else { -1 };
    if errorIdx >= 0 { Z(qs[errorIdx]); }
}

operation PhaseFlipCodeDemo(errorQubit : Int) : Result {
    use qs = Qubit[3];
    let alpha = 0.20;
    Ry(2.0 * ArcCos(Sqrt(alpha)), qs[0]);

    EncodePhaseFlip(qs[0], qs[1..2]);

    if errorQubit >= 0 { Z(qs[errorQubit]); }

    let syndrome = MeasurePhaseFlipSyndrome(qs);
    Message($"Phase-flip syndrome: {syndrome}");
    CorrectPhaseFlip(qs, syndrome);

    Adjoint EncodePhaseFlip(qs[0], qs[1..2]);
    let r = M(qs[0]);
    ResetAll(qs);
    r
}

In [ ]:
for err in [-1, 0, 1, 2]:
    label = "no error" if err < 0 else f"Z error on qubit {err}"
    results = qsharp.run(f"PhaseFlipCodeDemo({err})", 20)
    counts = Counter(str(r) for r in results)
    print(f"  {label:22s}: {dict(counts)}")

## 4.4 Shor's 9-Qubit Code

Shor's code concatenates the phase-flip code with the bit-flip code:  
1. Apply the phase-flip encoding (3 qubits)
2. Apply the bit-flip encoding to each of those 3 qubits (using 3 physical qubits each)
Result: 9 physical qubits protect 1 logical qubit against any single-qubit error.

In [ ]:
%%qsharp

import Std.Math.*;

/// Encode 1 qubit into Shor 9-qubit code
/// Layout: block0 = qs[0..2], block1 = qs[3..5], block2 = qs[6..8]
operation EncodeShor(qs : Qubit[]) : Unit is Adj {
    // Phase-flip layer: spread across blocks 0,1,2
    CNOT(qs[0], qs[3]);
    CNOT(qs[0], qs[6]);
    H(qs[0]); H(qs[3]); H(qs[6]);
    // Bit-flip layer: within each block
    CNOT(qs[0], qs[1]); CNOT(qs[0], qs[2]);
    CNOT(qs[3], qs[4]); CNOT(qs[3], qs[5]);
    CNOT(qs[6], qs[7]); CNOT(qs[6], qs[8]);
}

operation ShorCodeDemo(errorQubit : Int, errorType : Int) : Result {
    // errorType: 0=X, 1=Z, 2=Y, 3=none
    use qs = Qubit[9];

    // Prepare |+⟩ on the logical qubit
    H(qs[0]);

    EncodeShor(qs);

    // Inject error
    if errorType == 0 and errorQubit >= 0 { X(qs[errorQubit]); }
    elif errorType == 1 and errorQubit >= 0 { Z(qs[errorQubit]); }
    elif errorType == 2 and errorQubit >= 0 { Y(qs[errorQubit]); }

    // Decode (Adjoint of encoding)
    Adjoint EncodeShor(qs);

    // Measure logical qubit
    let r = M(qs[0]);
    ResetAll(qs);
    r
}

In [ ]:
# Test Shor code with X, Z, Y errors on various qubits
error_names = ["X", "Z", "Y", "none"]
print("Shor code recovery test (logical |+⟩ → measure in H basis, expect Zero):")
for etype, ename in zip([0, 1, 2], ["X", "Z", "Y"]):
    for eq in [0, 4, 8]:
        results = qsharp.run(f"ShorCodeDemo({eq}, {etype})", 30)
        counts = Counter(str(r) for r in results)
        print(f"  {ename} error on q{eq}: {dict(counts)}")

## 4.5 Repetition Code and Stabilizer Formalism

The stabilizer formalism describes a QEC code by its **stabilizer generators** — Pauli operators that leave all codewords invariant (eigenvalue +1). Errors anti-commute with some generators, producing a non-trivial syndrome.

For the 3-qubit bit-flip code, the stabilizers are:
- S₁ = Z₀Z₁ (checks parity of qubits 0 and 1)
- S₂ = Z₁Z₂ (checks parity of qubits 1 and 2)

An X error on qubit 0 anti-commutes with S₁ (gives syndrome 1,0). An X error on qubit 1 anti-commutes with both (syndrome 1,1).

In [ ]:
# Syndrome table for the 3-qubit bit-flip code
syndromes = {
    (0, 0): "No error",
    (1, 0): "X on qubit 0",
    (1, 1): "X on qubit 1",
    (0, 1): "X on qubit 2",
}
print("3-qubit bit-flip code syndrome table:")
print(f"{'S1=Z0Z1':>10}  {'S2=Z1Z2':>10}  {'Error':>15}")
print("-" * 40)
for (s1, s2), error in syndromes.items():
    print(f"{'One' if s1 else 'Zero':>10}  {'One' if s2 else 'Zero':>10}  {error:>15}")

## 4.6 QEC and Resource Estimation: Code Distance

Real-world QEC uses the **surface code**. The code distance `d` determines:
- Number of physical qubits per logical qubit: ~2d²
- Suppressed logical error rate: ~(p/p_th)^((d+1)/2)

where p is the physical error rate and p_th ≈ 1% is the threshold.

The resource estimator automatically chooses the minimum d needed to achieve a target logical error rate.

In [ ]:
# Visualise: physical qubits per logical qubit as a function of code distance
distances = np.arange(3, 25, 2)  # odd distances
phys_per_logical = 2 * distances**2

# Logical error rate suppression assuming p=0.1% physical, p_th=1%
p_phys = 1e-3
p_th = 1e-2
logical_error = [(p_phys / p_th) ** ((d + 1) / 2) for d in distances]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(distances, phys_per_logical, 'o-', color='steelblue')
ax1.set_xlabel("Code distance d")
ax1.set_ylabel("Physical qubits per logical qubit")
ax1.set_title("Surface code qubit overhead")
ax1.grid(alpha=0.4)

ax2.semilogy(distances, logical_error, 'o-', color='crimson')
ax2.set_xlabel("Code distance d")
ax2.set_ylabel("Logical error rate per round")
ax2.set_title("Error suppression (p_phys=0.1%, p_th=1%)")
ax2.grid(alpha=0.4, which='both')

plt.tight_layout()
plt.show()

print(f"At d=7:  {2*7**2} physical qubits/logical, error rate ≈ {(p_phys/p_th)**4:.2e}")
print(f"At d=15: {2*15**2} physical qubits/logical, error rate ≈ {(p_phys/p_th)**8:.2e}")

## 4.7 QEC Under Noisy Simulation

Let's verify that the bit-flip code actually improves fidelity under simulated noise.

In [ ]:
%%qsharp

import Std.Math.*;

// Bare qubit: prepare |1⟩, measure — any X error flips outcome
operation BareQubitMeasure() : Result {
    use q = Qubit();
    X(q);  // prepare |1⟩
    MResetZ(q)
}

// Protected qubit: encode |111⟩, correct, decode
operation ProtectedQubitMeasure() : Result {
    use qs = Qubit[3];
    X(qs[0]);
    // Encode
    CNOT(qs[0], qs[1]); CNOT(qs[0], qs[2]);
    // Syndrome measurement and correction (ancilla-based)
    use anc = Qubit[2];
    CNOT(qs[0], anc[0]); CNOT(qs[1], anc[0]);
    CNOT(qs[1], anc[1]); CNOT(qs[2], anc[1]);
    let (s01, s12) = (M(anc[0]), M(anc[1]));
    ResetAll(anc);
    let errIdx =
        if (s01, s12) == (One, Zero) { 0 }
        elif (s01, s12) == (One, One) { 1 }
        elif (s01, s12) == (Zero, One) { 2 }
        else { -1 };
    if errIdx >= 0 { X(qs[errIdx]); }
    // Decode
    CNOT(qs[0], qs[2]); CNOT(qs[0], qs[1]);
    let r = M(qs[0]);
    ResetAll(qs);
    r
}

In [ ]:
N = 500
noise_probs = [0.0, 0.01, 0.03, 0.05, 0.08, 0.10]

bare_errors = []
protected_errors = []

for p in noise_probs:
    noise = qsharp.BitFlipNoise(p) if p > 0 else None
    kwargs = dict(noise=noise) if noise else {}

    bare_r = qsharp.run("BareQubitMeasure()", N, **kwargs)
    prot_r = qsharp.run("ProtectedQubitMeasure()", N, **kwargs)

    # We prepared |1⟩ — Zero outcome = error
    bare_err = sum(1 for r in bare_r if str(r) == "Zero") / N
    prot_err = sum(1 for r in prot_r if str(r) == "Zero") / N
    bare_errors.append(bare_err)
    protected_errors.append(prot_err)
    print(f"p={p:.2f}  bare={bare_err:.3f}  protected={prot_err:.3f}")

plt.figure(figsize=(8, 4))
plt.plot(noise_probs, bare_errors, 'o-', label='Bare qubit')
plt.plot(noise_probs, protected_errors, 's--', label='Bit-flip code')
plt.xlabel("Bit-flip noise probability per gate")
plt.ylabel("Logical error rate")
plt.title("QEC effectiveness: bare vs 3-qubit bit-flip code")
plt.legend()
plt.grid(alpha=0.4)
plt.tight_layout()
plt.show()

## Summary

- **Bit-flip code**: 3 qubits, corrects one X error — syndrome from two parity checks (Z₀Z₁, Z₁Z₂)
- **Phase-flip code**: 3 qubits in H-basis, corrects one Z error — same structure in X-basis
- **Shor's 9-qubit code**: concatenates both, corrects any single-qubit error
- **Stabilizer formalism**: codes are defined by Pauli operators that stabilise all codewords
- **Code distance** determines the qubit overhead and error suppression: surface code uses ~2d² physical qubits per logical qubit
- The QDK resource estimator automatically selects the minimum code distance for a target error budget